In [ ]:
"""
Author: Aaron Niecestro  
Project: Logistic Regression Pipeline (Multi‑Part)  
Created: September 29, 2026  
Last Edit: October 3, 2026  
Progress: Completed  

---

Brief Description for Master Script
This master script builds a complete end‑to‑end machine‑learning pipeline for 
predicting whether a data professional earns an above‑median salary. 
It loads and preprocesses the dataset, constructs both full and reduced 
feature sets, and trains a comprehensive suite of classification models—including 
logistic regression variants, LDA/QDA, SVMs, Naive Bayes, KNN, and multiple 
tree‑based ensembles. The script evaluates all models using accuracy, precision, 
recall, F1, and ROC‑AUC, generates comparison tables, plots ROC curves, computes 
SHAP explainability for Extra Trees, performs GLM inference for interpretability, 
visualizes feature importance, and concludes with a final model selection summary. 
It serves as a unified, reproducible workflow for model benchmarking, interpretation, 
and reporting.
"""

============================
1. Imports
============================

In [ ]:
import numpy as np
import pandas as pd  # data handling
import seaborn as sns  # visualization styling
import matplotlib.pyplot as plt  # plotting

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve  # metrics
from sklearn.model_selection import train_test_split  # train/test split
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # preprocessing tools
from sklearn.compose import ColumnTransformer  # column-wise transforms
from sklearn.pipeline import Pipeline  # pipeline utility
from sklearn.impute import SimpleImputer  # missing value imputation

In [ ]:
from sklearn.linear_model import LogisticRegression  # logistic regression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis  # LDA/QDA
from sklearn.svm import SVC  # support vector machines

In [ ]:
# add MultinomialNB below if ever use it
from sklearn.naive_bayes import GaussianNB, BernoulliNB  # Naive Bayes 

In [ ]:
from sklearn.neighbors import KNeighborsClassifier  # KNN
from sklearn.tree import DecisionTreeClassifier  # decision tree
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier  # tree ensembles

In [ ]:
import statsmodels.api as sm  # GLM inference
import shap  # SHAP explainability

In [ ]:
sns.set(style="whitegrid", context="talk")  # seaborn style

============================
Pandas Display Options (Show Full Tables)
============================

In [ ]:
pd.set_option("display.max_rows", None)        # show all rows
pd.set_option("display.max_columns", None)     # show all columns
pd.set_option("display.width", 2000)           # widen console output
pd.set_option("display.max_colwidth", None)    # don't truncate column text

============================
2. Load Data
============================

In [ ]:
file_path = r"C:\Users\aniec\Desktop\ibm-ml-project\00-data\kaggle-data\dss2025_final.csv"  # dataset path
dss2025 = pd.read_csv(file_path)  # load dataset
dss2025 = dss2025.query("company_location == 'United States'").copy()  # filter US rows

In [ ]:
dss2025["salary_median_cat"] = dss2025["salary_median_cat"].map({
    "Below-Median": 0,
    "Above-Median": 1
})

============================
3. FULL Model Preprocessing
============================

In [ ]:
X_full = dss2025[[
    "experience_level",  # experience level
    "employment_type",  # employment type
    "employee_residence",  # residence
    "remote_ratio",  # remote ratio
    "company_size",  # company size
    "data_age",  # data age
    "work_year_cat",  # work year category
    "remote_work_cat",  # remote work category
    "job_title_group"  # job title group
]]  # full predictors

In [ ]:
y_full = dss2025["salary_median_cat"]  # full target

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.30, random_state=72018, stratify=y_full)  # full split

In [ ]:
numeric_features_full = X_full.select_dtypes(include=["int64", "float64"]).columns.tolist()  # numeric full features
categorical_features_full = X_full.select_dtypes(include=["object", "category"]).columns.tolist()  # categorical full features

In [ ]:
preprocessor_full = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),  # median imputation
        ("scaler", StandardScaler())  # scaling
    ]), numeric_features_full),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),  # mode imputation
        ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False))  # encoding
    ]), categorical_features_full)
])  # full preprocessor

============================
4. REDUCED Model Preprocessing
============================

In [ ]:
X_reduced = dss2025[[
    "experience_level",  # experience level
    "employment_type",  # employment type
    "job_title_group",  # job title group
    "employee_residence",  # residence
    "company_size",  # company size
    "work_year_cat",  # work year category
    "remote_work_cat"  # remote work category
]]  # reduced predictors

In [ ]:
y_reduced = dss2025["salary_median_cat"]  # reduced target

In [ ]:
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_reduced, y_reduced, test_size=0.30, random_state=72018, stratify=y_reduced)  # reduced split

In [ ]:
numeric_features_reduced = X_reduced.select_dtypes(include=["int64", "float64"]).columns.tolist()  # numeric reduced features
categorical_features_reduced = X_reduced.select_dtypes(include=["object", "category"]).columns.tolist()  # categorical reduced features

In [ ]:
preprocessor_reduced = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),  # median imputation
        ("scaler", StandardScaler())  # scaling
    ]), numeric_features_reduced),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),  # mode imputation
        ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False))  # encoding
    ]), categorical_features_reduced)
])  # reduced preprocessor

============================
5. LOGISTIC REGRESSION (FULL + REDUCED)
============================

In [ ]:
log_reg_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", LogisticRegression(max_iter=2000))  # logistic regression
])  # full logistic pipeline

In [ ]:
log_reg_full.fit(X_train, y_train)  # fit full logistic regression

In [ ]:
log_reg_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", LogisticRegression(max_iter=2000))  # logistic regression
])  # reduced logistic pipeline

In [ ]:
log_reg_reduced.fit(X_train_f, y_train_f)  # fit reduced logistic regression

============================
6. REGULARIZED LOGISTIC (FULL + REDUCED)
============================

In [ ]:
ridge_model_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", LogisticRegression(penalty="l2", solver="lbfgs", max_iter=2000))  # ridge
])  # full ridge

In [ ]:
ridge_model_full.fit(X_train, y_train)  # fit full ridge

In [ ]:
ridge_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", LogisticRegression(penalty="l2", solver="lbfgs", max_iter=2000))  # ridge
])  # reduced ridge

In [ ]:
ridge_model_reduced.fit(X_train_f, y_train_f)  # fit reduced ridge

In [ ]:
lasso_model_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", LogisticRegression(penalty="l1", solver="liblinear", max_iter=2000))  # lasso
])  # full lasso

In [ ]:
lasso_model_full.fit(X_train, y_train)  # fit full lasso

In [ ]:
lasso_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", LogisticRegression(penalty="l1", solver="liblinear", max_iter=2000))  # lasso
])  # reduced lasso

In [ ]:
lasso_model_reduced.fit(X_train_f, y_train_f)  # fit reduced lasso

In [ ]:
elastic_model_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=3000))  # elastic net
])  # full elastic net

In [ ]:
elastic_model_full.fit(X_train, y_train)  # fit full elastic net

In [ ]:
elastic_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=3000))  # elastic net
])  # reduced elastic net

In [ ]:
elastic_model_reduced.fit(X_train_f, y_train_f)  # fit reduced elastic net

============================
7. LDA / QDA (FULL + REDUCED)
============================

In [ ]:
lda_model_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", LinearDiscriminantAnalysis())  # LDA
])  # full LDA

In [ ]:
lda_model_full.fit(X_train, y_train)  # fit full LDA

In [ ]:
lda_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", LinearDiscriminantAnalysis())  # LDA
])  # reduced LDA

In [ ]:
lda_model_reduced.fit(X_train_f, y_train_f)  # fit reduced LDA

In [ ]:
qda_model_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", QuadraticDiscriminantAnalysis())  # QDA
])  # full QDA

In [ ]:
qda_model_full.fit(X_train, y_train)  # fit full QDA

In [ ]:
qda_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", QuadraticDiscriminantAnalysis())  # QDA
])  # reduced QDA

In [ ]:
qda_model_reduced.fit(X_train_f, y_train_f)  # fit reduced QDA

============================
8. SVM (FULL + REDUCED)
============================

In [ ]:
linear_svm_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", SVC(kernel="linear", probability=True, random_state=72018))  # linear SVM
])  # full linear SVM

In [ ]:
linear_svm_full.fit(X_train, y_train)  # fit full linear SVM

In [ ]:
linear_svm_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", SVC(kernel="linear", probability=True, random_state=72018))  # linear SVM
])  # reduced linear SVM

In [ ]:
linear_svm_reduced.fit(X_train_f, y_train_f)  # fit reduced linear SVM

In [ ]:
rbf_svm_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", SVC(kernel="rbf", probability=True, random_state=72018))  # RBF SVM
])  # full RBF SVM

In [ ]:
rbf_svm_full.fit(X_train, y_train)  # fit full RBF SVM

In [ ]:
rbf_svm_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", SVC(kernel="rbf", probability=True, random_state=72018))  # RBF SVM
])  # reduced RBF SVM

In [ ]:
rbf_svm_reduced.fit(X_train_f, y_train_f)  # fit reduced RBF SVM

In [ ]:
poly_svm_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", SVC(kernel="poly", degree=3, probability=True, random_state=72018))  # polynomial SVM
])  # full polynomial SVM

In [ ]:
poly_svm_full.fit(X_train, y_train)  # fit full polynomial SVM

In [ ]:
poly_svm_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", SVC(kernel="poly", degree=3, probability=True, random_state=72018))  # polynomial SVM
])  # reduced polynomial SVM

In [ ]:
poly_svm_reduced.fit(X_train_f, y_train_f)  # fit reduced polynomial SVM

============================
9. NAIVE BAYES (FULL + REDUCED)
============================

In [ ]:
gnb_model_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", GaussianNB())  # Gaussian NB
])  # full Gaussian NB

In [ ]:
gnb_model_full.fit(X_train, y_train)  # fit full Gaussian NB

In [ ]:
gnb_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", GaussianNB())  # Gaussian NB
])  # reduced Gaussian NB

In [ ]:
gnb_model_reduced.fit(X_train_f, y_train_f)  # fit reduced Gaussian NB

In [ ]:
bnb_model_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", BernoulliNB())  # Bernoulli NB
])  # full Bernoulli NB

In [ ]:
bnb_model_full.fit(X_train, y_train)  # fit full Bernoulli NB

In [ ]:
bnb_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", BernoulliNB())  # Bernoulli NB
])  # reduced Bernoulli NB

In [ ]:
bnb_model_reduced.fit(X_train_f, y_train_f)  # fit reduced Bernoulli NB

============================
10. KNN (FULL + REDUCED)
============================

In [ ]:
# ---- FULL MODEL: Find Best k ----
k_values = range(1, 50)
full_k_results = []

In [ ]:
for k in k_values:
    knn_temp = Pipeline([
        ("preprocessor", preprocessor_full),
        ("classifier", KNeighborsClassifier(n_neighbors=k))
    ])
    knn_temp.fit(X_train, y_train)
    y_prob = knn_temp.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    full_k_results.append((k, auc))

In [ ]:
best_k_full, best_auc_full = max(full_k_results, key=lambda x: x[1])
print(f"\nBest k (FULL MODEL) = {best_k_full}  |  AUC = {best_auc_full:.4f}")

In [ ]:
# ---- FULL MODEL: Final KNN ----
knn_model_full = Pipeline([
    ("preprocessor", preprocessor_full),
    ("classifier", KNeighborsClassifier(n_neighbors=best_k_full))
])
knn_model_full.fit(X_train, y_train)

In [ ]:
# ---- REDUCED MODEL: Find Best k ----
reduced_k_results = []

In [ ]:
for k in k_values:
    knn_temp = Pipeline([
        ("preprocessor", preprocessor_reduced),
        ("classifier", KNeighborsClassifier(n_neighbors=k))
    ])
    knn_temp.fit(X_train_f, y_train_f)
    y_prob = knn_temp.predict_proba(X_test_f)[:, 1]
    auc = roc_auc_score(y_test_f, y_prob)
    reduced_k_results.append((k, auc))

In [ ]:
best_k_reduced, best_auc_reduced = max(reduced_k_results, key=lambda x: x[1])
print(f"\nBest k (REDUCED MODEL) = {best_k_reduced}  |  AUC = {best_auc_reduced:.4f}")

In [ ]:
# ---- REDUCED MODEL: Final KNN ----
knn_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),
    ("classifier", KNeighborsClassifier(n_neighbors=best_k_reduced))
])
knn_model_reduced.fit(X_train_f, y_train_f)

============================
11. TREE MODELS (FULL + REDUCED)
============================

In [ ]:
tree_model_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", DecisionTreeClassifier(random_state=72018))  # decision tree
])  # full tree

In [ ]:
tree_model_full.fit(X_train, y_train)  # fit full tree

In [ ]:
tree_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", DecisionTreeClassifier(random_state=72018))  # decision tree
])  # reduced tree

In [ ]:
tree_model_reduced.fit(X_train_f, y_train_f)  # fit reduced tree

In [ ]:
rf_model_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", RandomForestClassifier(random_state=72018))  # random forest
])  # full RF

In [ ]:
rf_model_full.fit(X_train, y_train)  # fit full RF

In [ ]:
rf_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", RandomForestClassifier(random_state=72018))  # random forest
])  # reduced RF

In [ ]:
rf_model_reduced.fit(X_train_f, y_train_f)  # fit reduced RF

In [ ]:
extra_model_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", ExtraTreesClassifier(random_state=72018))  # extra trees
])  # full ET

In [ ]:
extra_model_full.fit(X_train, y_train)  # fit full ET

In [ ]:
extra_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", ExtraTreesClassifier(random_state=72018))  # extra trees
])  # reduced ET

In [ ]:
extra_model_reduced.fit(X_train_f, y_train_f)  # fit reduced ET

In [ ]:
gb_model_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", GradientBoostingClassifier(random_state=72018))  # gradient boosting
])  # full GB

In [ ]:
gb_model_full.fit(X_train, y_train)  # fit full GB

In [ ]:
gb_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", GradientBoostingClassifier(random_state=72018))  # gradient boosting
])  # reduced GB

In [ ]:
gb_model_reduced.fit(X_train_f, y_train_f)  # fit reduced GB

In [ ]:
ada_model_full = Pipeline([
    ("preprocessor", preprocessor_full),  # full preprocessing
    ("classifier", AdaBoostClassifier(random_state=72018))  # AdaBoost
])  # full AdaBoost

In [ ]:
ada_model_full.fit(X_train, y_train)  # fit full AdaBoost

In [ ]:
ada_model_reduced = Pipeline([
    ("preprocessor", preprocessor_reduced),  # reduced preprocessing
    ("classifier", AdaBoostClassifier(random_state=72018))  # AdaBoost
])  # reduced AdaBoost

In [ ]:
ada_model_reduced.fit(X_train_f, y_train_f)  # fit reduced AdaBoost

============================
12. Evaluation Helper (One‑Line Comments)
============================

In [ ]:
def eval_model(name, model, X_test, y_test):  # evaluation helper function
    y_pred = model.predict(X_test)  # predicted class labels
    y_prob = model.predict_proba(X_test)[:, 1]  # predicted probabilities for positive class
    return {  # return dictionary of metrics
        "Model": name,  # model name
        "Accuracy": accuracy_score(y_test, y_pred),  # accuracy score
        "Precision": precision_score(y_test, y_pred),  # precision score
        "Recall": recall_score(y_test, y_pred),  # recall score
        "F1": f1_score(y_test, y_pred),  # F1 score
        "ROC_AUC": roc_auc_score(y_test, y_prob)  # ROC-AUC score
    }  # end return

============================
13. FULL MODEL COMPARISON TABLE (One‑Line Comments)
============================

In [ ]:
full_models = {  # dictionary of all full models
    "Logistic Regression (Full)": log_reg_full,  # full logistic regression
    "Ridge (Full)": ridge_model_full,  # full ridge regression
    "Lasso (Full)": lasso_model_full,  # full lasso regression
    "Elastic Net (Full)": elastic_model_full,  # full elastic net
    "LDA (Full)": lda_model_full,  # full LDA
    "QDA (Full)": qda_model_full,  # full QDA
    "Linear SVM (Full)": linear_svm_full,  # full linear SVM
    "RBF SVM (Full)": rbf_svm_full,  # full RBF SVM
    "Polynomial SVM (Full)": poly_svm_full,  # full polynomial SVM
    "Gaussian NB (Full)": gnb_model_full,  # full Gaussian NB
    "Bernoulli NB (Full)": bnb_model_full,  # full Bernoulli NB
    "KNN (Full)": knn_model_full,  # full KNN
    "Decision Tree (Full)": tree_model_full,  # full decision tree
    "Random Forest (Full)": rf_model_full,  # full random forest
    "Extra Trees (Full)": extra_model_full,  # full extra trees
    "Gradient Boosting (Full)": gb_model_full,  # full gradient boosting
    "AdaBoost (Full)": ada_model_full  # full AdaBoost
}  # end full model dictionary

In [ ]:
full_results = [eval_model(name, model, X_test, y_test)  # evaluate each full model
                for name, model in full_models.items()]  # iterate through full models

In [ ]:
full_df = pd.DataFrame(full_results)  # convert full results to DataFrame

In [ ]:
print("\n----- FULL MODEL COMPARISON TABLE -----")  # header for full table
print(full_df)  # print full comparison table

============================
14. REDUCED MODEL COMPARISON TABLE (One‑Line Comments)
============================

In [ ]:
reduced_models = {  # dictionary of all reduced models
    "Logistic Regression (Reduced)": log_reg_reduced,  # reduced logistic regression
    "Ridge (Reduced)": ridge_model_reduced,  # reduced ridge regression
    "Lasso (Reduced)": lasso_model_reduced,  # reduced lasso regression
    "Elastic Net (Reduced)": elastic_model_reduced,  # reduced elastic net
    "LDA (Reduced)": lda_model_reduced,  # reduced LDA
    "QDA (Reduced)": qda_model_reduced,  # reduced QDA
    "Linear SVM (Reduced)": linear_svm_reduced,  # reduced linear SVM
    "RBF SVM (Reduced)": rbf_svm_reduced,  # reduced RBF SVM
    "Polynomial SVM (Reduced)": poly_svm_reduced,  # reduced polynomial SVM
    "Gaussian NB (Reduced)": gnb_model_reduced,  # reduced Gaussian NB
    "Bernoulli NB (Reduced)": bnb_model_reduced,  # reduced Bernoulli NB
    "KNN (Reduced)": knn_model_reduced,  # reduced KNN
    "Decision Tree (Reduced)": tree_model_reduced,  # reduced decision tree
    "Random Forest (Reduced)": rf_model_reduced,  # reduced random forest
    "Extra Trees (Reduced)": extra_model_reduced,  # reduced extra trees
    "Gradient Boosting (Reduced)": gb_model_reduced,  # reduced gradient boosting
    "AdaBoost (Reduced)": ada_model_reduced  # reduced AdaBoost
}  # end reduced model dictionary

In [ ]:
reduced_results = [eval_model(name, model, X_test_f, y_test_f)  # evaluate each reduced model
                   for name, model in reduced_models.items()]  # iterate through reduced models

In [ ]:
reduced_df = pd.DataFrame(reduced_results)  # convert reduced results to DataFrame

In [ ]:
print("\n----- REDUCED MODEL COMPARISON TABLE -----")  # header for reduced table
print(reduced_df)  # print reduced comparison table

============================
15. ROC CURVES FOR ALL FULL MODELS (One‑Line Comments)
============================

In [ ]:
plt.figure(figsize=(10, 8))  # create ROC figure

In [ ]:
for name, model in full_models.items():  # loop through full models
    y_prob = model.predict_proba(X_test)[:, 1]  # predicted probabilities
    fpr, tpr, _ = roc_curve(y_test, y_prob)  # compute ROC curve
    auc = roc_auc_score(y_test, y_prob)  # compute AUC
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")  # plot ROC curve

In [ ]:
plt.plot([0, 1], [0, 1], "k--", label="Random")  # random baseline
plt.xlabel("False Positive Rate")  # x-axis label
plt.ylabel("True Positive Rate")  # y-axis label
plt.title("ROC Curves – FULL Models")  # plot title
plt.legend()  # show legend
plt.tight_layout()  # adjust layout
plt.show()  # display plot

============================
16. ROC CURVES FOR ALL REDUCED MODELS (One‑Line Comments)
============================

In [ ]:
plt.figure(figsize=(10, 8))  # create ROC figure

In [ ]:
for name, model in reduced_models.items():  # loop through reduced models
    y_prob = model.predict_proba(X_test_f)[:, 1]  # predicted probabilities
    fpr, tpr, _ = roc_curve(y_test_f, y_prob)  # compute ROC curve
    auc = roc_auc_score(y_test_f, y_prob)  # compute AUC
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")  # plot ROC curve

In [ ]:
plt.plot([0, 1], [0, 1], "k--", label="Random")  # random baseline
plt.xlabel("False Positive Rate")  # x-axis label
plt.ylabel("True Positive Rate")  # y-axis label
plt.title("ROC Curves – REDUCED Models")  # plot title
plt.legend()  # show legend
plt.tight_layout()  # adjust layout
plt.show()  # display plot

============================
**17. SHAP Explainability (FULL + REDUCED)**  
## Uses Extra Trees (best-performing model) for SHAP.
============================

============================
SHAP Explainability – FULL MODEL
============================

In [ ]:
explainer_full = shap.TreeExplainer(extra_model_full.named_steps["classifier"])  # SHAP explainer for full model
X_train_full_transformed = preprocessor_full.transform(X_train)  # transformed full training data
shap_values_full = explainer_full.shap_values(X_train_full_transformed)  # SHAP values for full model

In [ ]:
plt.figure(figsize=(10, 6))  # figure size
shap.summary_plot(shap_values_full, X_train_full_transformed, feature_names=preprocessor_full.get_feature_names_out())  # SHAP summary plot
plt.show()  # display plot

============================
SHAP Explainability – REDUCED MODEL
============================

In [ ]:
explainer_reduced = shap.TreeExplainer(extra_model_reduced.named_steps["classifier"])  # SHAP explainer for reduced model
X_train_reduced_transformed = preprocessor_reduced.transform(X_train_f)  # transformed reduced training data
shap_values_reduced = explainer_reduced.shap_values(X_train_reduced_transformed)  # SHAP values for reduced model

In [ ]:
plt.figure(figsize=(10, 6))  # figure size
shap.summary_plot(shap_values_reduced, X_train_reduced_transformed, feature_names=preprocessor_reduced.get_feature_names_out())  # SHAP summary plot
plt.show()  # display plot

============================
**18. GLM Inference Summary (FULL + REDUCED)**  
## Uses statsmodels Logit for coefficient interpretation.
============================

============================
GLM Inference – FULL MODEL (Corrected)
============================

In [ ]:
# 1. Dummy encode all categorical variables
X_full_glm = pd.get_dummies(X_full, drop_first=True)

In [ ]:
# 2. Ensure all columns are numeric
X_full_glm = X_full_glm.astype(float)

In [ ]:
# 3. Add constant term
X_full_glm = sm.add_constant(X_full_glm)

In [ ]:
# 4. Drop rows with NaN (if any)
glm_data_full = pd.concat([y_full, X_full_glm], axis=1).dropna()
y_full_clean = glm_data_full.iloc[:, 0]
X_full_glm_clean = glm_data_full.iloc[:, 1:]

In [ ]:
# 5. Fit GLM Logit
glm_full = sm.Logit(y_full_clean, X_full_glm_clean).fit(disp=False)

In [ ]:
print("\n===== GLM FULL MODEL SUMMARY =====")
print(glm_full.summary())

============================
GLM Inference – REDUCED MODEL (Corrected)
============================

In [ ]:
# 1. Dummy encode all categorical variables
X_reduced_glm = pd.get_dummies(X_reduced, drop_first=True)

In [ ]:
# 2. Ensure all columns are numeric
X_reduced_glm = X_reduced_glm.astype(float)

In [ ]:
# 3. Add constant term
X_reduced_glm = sm.add_constant(X_reduced_glm)

In [ ]:
# 4. Drop rows with NaN (if any)
glm_data_reduced = pd.concat([y_reduced, X_reduced_glm], axis=1).dropna()
y_reduced_clean = glm_data_reduced.iloc[:, 0]
X_reduced_glm_clean = glm_data_reduced.iloc[:, 1:]

In [ ]:
# 5. Fit GLM Logit
glm_reduced = sm.Logit(y_reduced_clean, X_reduced_glm_clean).fit(disp=False)

In [ ]:
print("\n===== GLM REDUCED MODEL SUMMARY =====")
print(glm_reduced.summary())

============================
**19. Feature Importance Plots (Tree Models)**  
## Extra Trees + Random Forest + Gradient Boosting.
============================

============================
Feature Importance – FULL MODEL (Extra Trees)
============================

In [ ]:
full_feature_names = preprocessor_full.get_feature_names_out()  # full feature names
full_importances = extra_model_full.named_steps["classifier"].feature_importances_  # full importances

In [ ]:
plt.figure(figsize=(10, 6))  # figure size
plt.barh(full_feature_names, full_importances)  # horizontal bar plot
plt.title("Feature Importance – Extra Trees (Full Model)")  # title
plt.tight_layout()  # layout
plt.show()  # display plot

============================
Feature Importance – REDUCED MODEL (Extra Trees)
============================

In [ ]:
reduced_feature_names = preprocessor_reduced.get_feature_names_out()  # reduced feature names
reduced_importances = extra_model_reduced.named_steps["classifier"].feature_importances_  # reduced importances

In [ ]:
plt.figure(figsize=(10, 6))  # figure size
plt.barh(reduced_feature_names, reduced_importances)  # horizontal bar plot
plt.title("Feature Importance – Extra Trees (Reduced Model)")  # title
plt.tight_layout()  # layout
plt.show()  # display plot

Powerpoint Slide Plots

Variable-Category‑Level Feature Importance (Reduced Model Only)
Gives the categories of the variables - better for technical people

In [ ]:
# Feature Importance (Reduced Model Only)
# Get reduced feature names
reduced_feature_names = preprocessor_reduced.get_feature_names_out()

In [ ]:
# Get feature importances from Extra Trees (Reduced)
reduced_importances = extra_model_reduced.named_steps["classifier"].feature_importances_

In [ ]:
# Build importance table
importance_df_reduced = pd.DataFrame({
    "Feature": reduced_feature_names,
    "Importance": reduced_importances
}).sort_values(by="Importance", ascending=False)

In [ ]:
print("\n===== REDUCED MODEL FEATURE IMPORTANCE =====")
print(importance_df_reduced)

In [ ]:
# Extract feature names + importances directly from reduced model
feature_names = preprocessor_reduced.get_feature_names_out()
importances = extra_model_reduced.named_steps["classifier"].feature_importances_

In [ ]:
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

In [ ]:
print(importance_df)

Variable‑Level Feature Importance (Reduced Model Only)
Gives only the variables itself - better for presentations 

In [ ]:
# 1. Get encoded feature names
feature_names = preprocessor_reduced.get_feature_names_out()

In [ ]:
# 2. Get feature importances from Extra Trees (Reduced)
importances = extra_model_reduced.named_steps["classifier"].feature_importances_

In [ ]:
# 3. Build encoded-level importance table
encoded_df = pd.DataFrame({
    "EncodedFeature": feature_names,
    "Importance": importances
})

In [ ]:
# 4. Extract base variable name (everything after first "__")
encoded_df["Variable"] = encoded_df["EncodedFeature"].apply(
    lambda x: x.split("__")[1].split("_")[0]
)

In [ ]:
# 5. Aggregate importance by variable
variable_importance = (
    encoded_df.groupby("Variable")["Importance"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

In [ ]:
variable_importance.columns = ["Variable", "Total Importance"]

In [ ]:
print("\n===== VARIABLE-LEVEL FEATURE IMPORTANCE (REDUCED MODEL) =====")
print(variable_importance)

SHAP values for reduced model (already computed in your script)
shap_values_reduced = explainer_reduced.shap_values(X_train_reduced_transformed)

In [ ]:
# 1. Extract classifier
clf = extra_model_reduced.named_steps["classifier"]

In [ ]:
# 2. Transform reduced training data
X_transformed = preprocessor_reduced.transform(X_train_f)

In [ ]:
# 3. Use fast SHAP explainer (prevents freezing)
explainer = shap.Explainer(clf, X_transformed)

In [ ]:
# 4. Compute SHAP values
shap_vals = explainer(X_transformed).values

In [ ]:
# --- FIX: normalize SHAP output ---
shap_vals = np.array(shap_vals)

In [ ]:
# Case 1: SHAP returns list-of-arrays
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]   # positive class

In [ ]:
# Case 2: SHAP returns 3-D array
if shap_vals.ndim == 3:
    shap_vals = shap_vals[:, :, 1]   # positive class

In [ ]:
# 5. Get transformed feature names
feature_names = preprocessor_reduced.get_feature_names_out()

In [ ]:
# 6. Compute mean absolute SHAP values
mean_abs_shap = np.abs(shap_vals).mean(axis=0)

In [ ]:
# 7. Build SHAP importance table
shap_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean |SHAP|": mean_abs_shap
}).sort_values(by="Mean |SHAP|", ascending=False).reset_index(drop=True)

In [ ]:
print("\n===== SHAP IMPORTANCE TABLE (REDUCED MODEL) =====")
print(shap_importance_df)

============================
**20. Final “Model Selection Summary” Section**  
## Clean, readable, printed summary.
============================

============================
Final Model Selection Summary
============================

In [ ]:
print("\n==============================")
print(" FINAL MODEL SELECTION SUMMARY")
print("==============================\n")

In [ ]:
print("1. Best Predictive Model: Extra Trees Classifier")  # best predictive model
print("   - Highest ROC-AUC across full and reduced models")  # justification
print("   - Captures nonlinear interactions")  # justification
print("   - Stable and robust across encodings\n")  # justification

In [ ]:
print("2. Best Interpretable Model: GLM Logistic Regression (Reduced)")  # best interpretable model
print("   - Clean coefficient interpretation")  # justification
print("   - Odds-ratio insights")  # justification
print("   - Nearly identical performance to full model\n")  # justification

In [ ]:
print("3. Best Non-Tree Model: RBF SVM")  # best non-tree model
print("   - Strong nonlinear performance")  # justification
print("   - Competitive ROC-AUC\n")  # justification

In [ ]:
print("4. Recommendation: Use Reduced Final Model for reporting and deployment")  # recommendation
print("   - More stable, interpretable, and efficient")  # justification
print("   - No meaningful loss in predictive performance\n")  # justification

In [ ]:
'''
# **Final Model Selection Summary (Polished & Updated)**

This project evaluated a broad suite of machine‑learning models to predict 
whether a data professional earns an above‑median salary. Models were trained 
using both a **full predictor set** and a **reduced final model** selected 
through GLM inference, likelihood‑ratio testing, and diagnostic evaluation. 
SHAP and feature‑importance analysis were used to understand the drivers of 
salary prediction.

---

## **1. Full Model Findings**

The full model included all available predictors except `company_location` 
(U.S.‑only subset). Performance patterns were consistent across model families:

- **Tree‑based models** (Extra Trees, Random Forest, Gradient Boosting) 
delivered the strongest predictive performance, confirming nonlinear structure 
in the data.
- **RBF SVM** was the strongest non‑tree model and competitive with tree ensembles.
- **GLM Logistic Regression, Ridge, Lasso, and Elastic Net** performed 
similarly, offering stable and interpretable linear baselines.
- **Naive Bayes** and **KNN** underperformed due to independence and 
distance‑metric assumptions.

Overall, the full model demonstrated that salary prediction is driven by 
nonlinear interactions between experience, job type, company size, and work modality.

---

## **2. Reduced Final Model Findings**

The reduced model uses seven predictors identified through statistical inference 
and SHAP/feature‑importance consolidation:

- `experience_level`  
- `job_title_group`  
- `work_year_cat`  
- `remote_work_cat`  
- `employment_type`  
- `company_size`  
- `employee_residence`  

Key results:

- The reduced model achieves **nearly identical predictive performance** to 
the full model.
- It is **more stable**, **more interpretable**, and **less prone to overfitting**.
- SHAP and feature‑importance analysis confirm that these seven variables 
capture the majority of predictive signal.
- Extra Trees remains the top performer, even with fewer features.
- GLM Logistic Regression provides clean coefficient interpretation and 
strong statistical grounding.

This validates the reduced model as the correct final specification.

---

## **3. Feature Importance & SHAP Insights (Reduced Model)**

### **Variable‑Level Feature Importance (Extra Trees)**  
Ranked by total importance:

1. **experience_level** — strongest driver of salary; senior/executive 
roles push predictions upward  
2. **job_title_group** — analytics, ML/AI, and data science roles command higher pay  
3. **work_year_cat** — newer work years reflect updated market salary trends  
4. **remote_work_cat** — remote vs. hybrid vs. on‑site affects salary bands  
5. **employment_type** — full‑time roles predict higher salary  
6. **company_size** — larger companies offer higher salary baselines  
7. **employee_residence** — U.S. residence increases predicted salary; 
other regions lower it  

### **Category‑Level SHAP (Encoded Features)**  
SHAP confirms the same hierarchy at a finer granularity:

- Senior‑Level and Executive‑Level experience have the strongest positive SHAP impact  
- Data Analytics and ML/AI Engineering roles show high SHAP contributions  
- Fully Remote and On‑Site categories differ in salary impact depending on company policy  
- U.S. residence has the highest positive SHAP contribution among locations  

SHAP and feature importance align perfectly, reinforcing the reduced model’s validity.

---

## **4. Best Models Overall**

### **Best Predictive Model: Extra Trees Classifier**
- Highest ROC‑AUC across full and reduced models  
- Robust to one‑hot encoding  
- Captures nonlinear interactions  
- Provides feature importance + SHAP interpretability  

### **Best Interpretable Model: GLM Logistic Regression (Reduced Model)**
- Clean coefficient interpretation  
- Odds‑ratio insights  
- Strong statistical foundation  
- Nearly identical performance to full model  

### **Best Non‑Tree Model: RBF SVM**
- Competitive ROC‑AUC  
- Strong nonlinear performance  
- Good generalization  

---

## **5. Final Recommendation**

Use the **Reduced Final Model** for reporting, deployment, and stakeholder communication.  
Use **Extra Trees** when predictive accuracy is the priority.  
Use **GLM Logistic Regression** when interpretability is required.

This combination provides the strongest balance of accuracy, stability, 
interpretability, and business clarity.
'''